In [1]:
from model.PicSure import PicSureS
import torch
import torchvision
import torchvision.transforms as transforms
import os
import random
from PIL import Image

/Users/cowolff/miniconda3/envs/dfki/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/cowolff/miniconda3/envs/dfki/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
# Set the root directory for CIFAR dataset
dataset_root = "./data"
output_root = "./CIFAR"

# Define transformation (to convert to tensor and normalize)
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load the CIFAR-10 dataset
dataset = torchvision.datasets.CIFAR10(root=dataset_root, train=True, download=True, transform=transform)

# Get class names
classes = dataset.classes

# Create output directory if it doesn't exist
os.makedirs(output_root, exist_ok=True)

# Dictionary to store selected images
selected_images = {class_name: [] for class_name in classes}

# Randomly shuffle indices
indices = list(range(len(dataset)))
random.shuffle(indices)

# Collect 10 images per class
for idx in indices:
    image, label = dataset[idx]
    class_name = classes[label]

    if len(selected_images[class_name]) < 10:
        selected_images[class_name].append(image)

    # Stop if all classes have 10 images
    if all(len(imgs) == 10 for imgs in selected_images.values()):
        break

# Save images to class folders
for class_name, images in selected_images.items():
    class_dir = os.path.join(output_root, class_name)
    os.makedirs(class_dir, exist_ok=True)

    for i, image in enumerate(images):
        # Convert tensor to PIL image
        pil_image = transforms.ToPILImage()(image)
        image_path = os.path.join(class_dir, f"{i}.png")
        pil_image.save(image_path)

print("Images successfully saved in CIFAR directory!")


Files already downloaded and verified
Images successfully saved in CIFAR directory!


In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = PicSureS(num_classes=10, download=True).to(device)

Model weights already exist, skipping download.


/Users/cowolff/miniconda3/envs/dfki/lib/python3.10/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [4]:
def load_cifar_images_to_dict(root_dir="./CIFAR", device="cpu"):
    """
    Loads images from the CIFAR directory into a dictionary.

    Args:
        root_dir (str): The path to the CIFAR directory.

    Returns:
        dict: A dictionary where keys are class names and values are lists of images as PyTorch tensors.
    """
    transform = transforms.ToTensor()
    image_dict = {}

    # Iterate through class directories
    for class_name in os.listdir(root_dir):
        class_path = os.path.join(root_dir, class_name)

        # Ensure it's a directory
        if os.path.isdir(class_path):
            image_dict[class_name] = []

            # Load each image in the class folder
            for image_file in sorted(os.listdir(class_path)):  # Sorting ensures consistent ordering
                image_path = os.path.join(class_path, image_file)

                # Open and convert to tensor
                image = Image.open(image_path)
                image_tensor = transform(image).to(device)

                image_dict[class_name].append(image_tensor)

    return image_dict

def get_random_image_label(root_dir="./CIFAR", device="cpu"):
    """
    Randomly selects an image and its label from the CIFAR directory.

    Args:
        root_dir (str): The path to the CIFAR directory.

    Returns:
        tuple: A tuple containing the image as a PyTorch tensor and its label.
    """
    image_dict = load_cifar_images_to_dict(root_dir, device)

    # Choose a random class
    class_name = random.choice(list(image_dict.keys()))
    images = image_dict[class_name]

    # Choose a random image from the class
    image = random.choice(images)

    # Get the label index
    label = classes.index(class_name)

    return image, label

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = PicSureS(num_classes=10, download=True).to(device)

cifar_images = load_cifar_images_to_dict(device=device)

model.setContextImages(cifar_images)

# Get a random image and label
image, label = get_random_image_label(device=device)

# Perform inference
output = model.predict(image.unsqueeze(0))

/Users/cowolff/Documents/GitHub/embed-then-classify/model/PicSure.py:78: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  images = torch.tensor(images)
/Users/cowolff/Documents/GitHub/embed-then-classify/model/PicSure.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.context_images[class_name] = torch.tensor(images)


RuntimeError: shape '[1, 3, 512]' is invalid for input of size 512